In [ ]:
!pip install roboflow ultralytics

from roboflow import Roboflow
rf = Roboflow(api_key="api_key")
project = rf.workspace("js-ozptv").project("foggy-car-ofcf4")
version = project.version(1)
dataset = version.download("yolo26")


In [ ]:
print(dataset.location)

In [ ]:
DATASET_PATH = dataset.location

In [ ]:
import os
from collections import defaultdict

def dataset_sanity_check(dataset_path, split="train"):
    images_dir = os.path.join(dataset_path, split, "images")
    labels_dir = os.path.join(dataset_path, split, "labels")

    image_files = set(os.listdir(images_dir))
    label_files = set(os.listdir(labels_dir))

    image_basenames = {os.path.splitext(f)[0] for f in image_files}
    label_basenames = {os.path.splitext(f)[0] for f in label_files}

    missing_labels = image_basenames - label_basenames
    orphan_labels = label_basenames - image_basenames

    empty_labels = []
    total_boxes = 0

    for lbl in label_files:
        with open(os.path.join(labels_dir, lbl)) as f:
            lines = f.readlines()
            if len(lines) == 0:
                empty_labels.append(lbl)
            total_boxes += len(lines)

    print(f" Split: {split}")
    print(f" Images: {len(image_files)}")
    print(f" Label files: {len(label_files)}")
    print(f" Total bounding boxes: {total_boxes}")
    print(f" Images without labels: {len(missing_labels)}")
    print(f" Orphan label files: {len(orphan_labels)}")
    print(f" Empty label files: {len(empty_labels)}")

    return {
        "missing_labels": missing_labels,
        "orphan_labels": orphan_labels,
        "empty_labels": empty_labels
    }


In [ ]:
for split in ["train", "valid", "test"]:
    dataset_sanity_check(DATASET_PATH, split)


In [ ]:
import matplotlib.pyplot as plt

def class_distribution(dataset_path, split="train"):
    labels_dir = os.path.join(dataset_path, split, "labels")
    class_counts = defaultdict(int)

    for lbl in os.listdir(labels_dir):
        with open(os.path.join(labels_dir, lbl)) as f:
            for line in f:
                class_id = int(line.split()[0])
                class_counts[class_id] += 1

    return class_counts


def plot_class_distribution(class_counts, title):
    classes = list(class_counts.keys())
    counts = list(class_counts.values())

    plt.figure(figsize=(8, 4))
    plt.bar(classes, counts)
    plt.xlabel("Class ID")
    plt.ylabel("Number of Boxes")
    plt.title(title)
    plt.show()


In [ ]:
train_counts = class_distribution(DATASET_PATH, "train")
plot_class_distribution(train_counts, "Train Class Distribution")


In [ ]:
import numpy as np

def bbox_statistics(dataset_path, split="train"):
    labels_dir = os.path.join(dataset_path, split, "labels")

    widths, heights, areas = [], [], []

    for lbl in os.listdir(labels_dir):
        with open(os.path.join(labels_dir, lbl)) as f:
            for line in f:
                _, _, _, w, h = map(float, line.split())
                widths.append(w)
                heights.append(h)
                areas.append(w * h)

    return np.array(widths), np.array(heights), np.array(areas)


In [ ]:
w, h, area = bbox_statistics(DATASET_PATH, "train")

plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.hist(w, bins=50)
plt.title("BBox Width")

plt.subplot(1,3,2)
plt.hist(h, bins=50)
plt.title("BBox Height")

plt.subplot(1,3,3)
plt.hist(area, bins=50)
plt.title("BBox Area")

plt.show()


In [ ]:
def object_size_ratio(areas):
    small = np.sum(areas < 0.01)
    medium = np.sum((areas >= 0.01) & (areas < 0.05))
    large = np.sum(areas >= 0.05)

    total = len(areas)

    print(f"Small objects:  {small/total:.2%}")
    print(f"Medium objects: {medium/total:.2%}")
    print(f"Large objects:  {large/total:.2%}")


In [ ]:
object_size_ratio(area)


In [ ]:
import cv2
from tqdm import tqdm

def fog_contrast_scores(dataset_path, split="train", sample=300):
    images_dir = os.path.join(dataset_path, split, "images")
    scores = []

    for img_name in tqdm(os.listdir(images_dir)[:sample]):
        img = cv2.imread(os.path.join(images_dir, img_name), cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        scores.append(img.std())  # lower std = heavier fog

    return scores


In [ ]:
fog_scores = fog_contrast_scores(DATASET_PATH, "train")

plt.hist(fog_scores, bins=50)
plt.xlabel("Contrast (std)")
plt.ylabel("Image Count")
plt.title("Fog Density Distribution")
plt.show()


In [ ]:
import os
import random
import matplotlib.pyplot as plt
import cv2
import numpy as np
import math


def draw_yolo_boxes(img, label_path, class_names=None):
    h, w = img.shape[:2]

    if not os.path.exists(label_path):
        return img

    with open(label_path, "r") as f:
        lines = f.readlines()

    for line in lines:
        cls, xc, yc, bw, bh = map(float, line.split())

        # Convert normalized → pixel coords
        x1 = int((xc - bw / 2) * w)
        y1 = int((yc - bh / 2) * h)
        x2 = int((xc + bw / 2) * w)
        y2 = int((yc + bh / 2) * h)

        color = (0, 255, 0)  # green box
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)

        if class_names:
            label = class_names[int(cls)]
            cv2.putText(
                img,
                label,
                (x1, max(y1 - 5, 15)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                color,
                2,
            )

    return img

def show_random_samples_with_boxes(
    dataset_path,
    split="train",
    n=5,
    class_names=None,
    cols=3  # 👈 number of images per row
):
    images_dir = os.path.join(dataset_path, split, "images")
    labels_dir = os.path.join(dataset_path, split, "labels")

    samples = random.sample(os.listdir(images_dir), n)

    rows = math.ceil(n / cols)

    # BIG figure size
    plt.figure(figsize=(6 * cols, 6 * rows))

    for i, img_name in enumerate(samples):
        img_path = os.path.join(images_dir, img_name)
        label_path = os.path.join(
            labels_dir,
            os.path.splitext(img_name)[0] + ".txt"
        )

        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        img = draw_yolo_boxes(img, label_path, class_names)

        plt.subplot(rows, cols, i + 1)
        plt.imshow(img)
        plt.axis("off")
        plt.title(img_name, fontsize=10)

    plt.tight_layout()
    plt.show()


In [ ]:
import yaml

with open("/content/foggy-car-1/data.yaml") as f:
    data = yaml.safe_load(f)

class_names = data["names"]


In [ ]:
class_names

In [ ]:
show_random_samples_with_boxes(
    DATASET_PATH,
    split="train",
    n=5,
    class_names=class_names
)


In [ ]:
from ultralytics import YOLO
from ultralytics.data.dataset import YOLODataset
import ultralytics.data.build as build
import numpy as np
import matplotlib.pyplot as plt
import cv2

In [ ]:
import os
import shutil

train_images = "/content/foggy-car-1/train/images"
train_labels = "/content/foggy-car-1/train/labels"

# bus and truck
target_classes = [1, 4]

duplicate_factor = 1  # 1 means duplicate once (2x total)

for label_file in os.listdir(train_labels):
    label_path = os.path.join(train_labels, label_file)

    with open(label_path, "r") as f:
        lines = f.readlines()

    contains_target = any(
        int(line.split()[0]) in target_classes
        for line in lines
    )

    if contains_target:
        image_file = label_file.replace(".txt", ".jpg")

        for i in range(duplicate_factor):
            new_image_name = image_file.replace(".jpg", f"_dup{i}.jpg")
            new_label_name = label_file.replace(".txt", f"_dup{i}.txt")

            shutil.copy(
                os.path.join(train_images, image_file),
                os.path.join(train_images, new_image_name)
            )
            shutil.copy(
                os.path.join(train_labels, label_file),
                os.path.join(train_labels, new_label_name)
            )

print("Duplication complete.")


In [ ]:
class YOLOWeightedDataset(YOLODataset):
    def __init__(self, *args, mode="train", **kwargs):
        """
        Initialize the WeightedDataset.

        Args:
            class_weights (list or numpy array): A list or array of weights corresponding to each class.
        """

        super(YOLOWeightedDataset, self).__init__(*args, **kwargs)

        self.train_mode = "train" in self.prefix

        # You can also specify weights manually instead
        self.count_instances()
        class_weights = np.sum(self.counts) / self.counts
        self.agg_func = np.mean

        self.class_weights = np.array(class_weights)
        self.weights = self.calculate_weights()
        self.probabilities = self.calculate_probabilities()

    def count_instances(self):
        """
        Count the number of instances per class

        Returns:
            dict: A dict containing the counts for each class.
        """
        self.counts = [0 for i in range(len(self.data["names"]))]
        for label in self.labels:
            cls = label['cls'].reshape(-1).astype(int)
            for id in cls:
                self.counts[id] += 1

        self.counts = np.array(self.counts)
        self.counts = np.where(self.counts == 0, 1, self.counts)

    def calculate_weights(self):
        """
        Calculate the aggregated weight for each label based on class weights.

        Returns:
            list: A list of aggregated weights corresponding to each label.
        """
        weights = []
        for label in self.labels:
            cls = label['cls'].reshape(-1).astype(int)

            # Give a default weight to background class
            if cls.size == 0:
              weights.append(1)
              continue

            # Take mean of weights
            # You can change this weight aggregation function to aggregate weights differently
            # weight = np.mean(self.class_weights[cls])
            # weight = np.max(self.class_weights[cls])
            weight = self.agg_func(self.class_weights[cls])
            weights.append(weight)
        return weights

    def calculate_probabilities(self):
        """
        Calculate and store the sampling probabilities based on the weights.

        Returns:
            list: A list of sampling probabilities corresponding to each label.
        """
        total_weight = sum(self.weights)
        probabilities = [w / total_weight for w in self.weights]
        return probabilities

    def __getitem__(self, index):
        """
        Return transformed label information based on the sampled index.
        """
        # Don't use for validation
        if not self.train_mode:
            return self.transforms(self.get_image_and_label(index))
        else:
            index = np.random.choice(len(self.labels), p=self.probabilities)
            return self.transforms(self.get_image_and_label(index))

In [ ]:
build.YOLODataset = YOLOWeightedDataset

In [ ]:
import torch

#  Recommended for YOLO training
torch.backends.cuda.matmul.fp32_precision = "tf32"
torch.backends.cudnn.conv.fp32_precision = "tf32"

# Optional: enable cudnn benchmark for speed
torch.backends.cudnn.benchmark = True


In [ ]:
model = YOLO("yolo26m.pt")

In [ ]:
results = model.train(
    data="/content/foggy-car-1/data.yaml",
    epochs=120,
    imgsz=640,

    # Batch
    batch=96,  # Slightly lower than 128 for better gradient stability

    # Optimizer
    optimizer="SGD",          # More stable for detection
    lr0=0.01,                 # Proper SGD default
    lrf=0.1,                  # Higher final LR (less aggressive decay)
    momentum=0.937,
    weight_decay=5e-4,
    cos_lr=True,

    # Warmup
    warmup_epochs=5.0,        # Slightly longer warmup
    warmup_bias_lr=0.05,      # Lower than 0.1 for smoother start
    warmup_momentum=0.8,

    # Augmentations
    mosaic=1.0,
    close_mosaic=20,          # Disable earlier for stability
    copy_paste=0.1,           # Reduce from 0.2 (too aggressive)
    hsv_h=0.015,
    hsv_s=0.4,
    hsv_v=0.4,
    scale=0.5,
    translate=0.1,

    # Loss balancing (important for recall)
    box=9.0,                  # Increase box weight (better localization → better recall)
    cls=0.5,
    dfl=1.5,

    # Performance
    cache="disk",             # Avoid RAM nondeterminism
    amp=True,
    compile=True,
    pretrained=True,

    # Stability
    freeze=0,                 # Let backbone adapt to fog
    workers=8,
    patience=30,              # Give model more room
)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# plt.figure(figsize=(10,5))
# plt.plot(results['metrics/mAP50(B)'], label='mAP@50')
# plt.plot(results['metrics/mAP50-95(B)'], label='mAP@50-95')
# plt.xlabel("Epoch")
# plt.ylabel("mAP")
# plt.legend()
# plt.title("mAP over Epochs")
# plt.show()


In [ ]:
# plt.figure(figsize=(10,5))
# plt.plot(results['train/box_loss'], label='Train Box Loss')
# plt.plot(results['val/box_loss'], label='Val Box Loss')
# plt.legend()
# plt.title("Box Loss")
# plt.show()


In [ ]:
# import pandas as pd
# import matplotlib.pyplot as plt

# # Load results
# results = pd.read_csv("runs/detect/train/results.csv")

# # Remove whitespace in column names (important)
# results.columns = results.columns.str.strip()

# # Plot Precision & Recall
# plt.figure(figsize=(10,5))
# plt.plot(results['metrics/precision(B)'], label='Precision')
# plt.plot(results['metrics/recall(B)'], label='Recall')
# plt.xlabel("Epoch")
# plt.ylabel("Score")
# plt.legend()
# plt.title("Precision vs Recall")
# plt.show()


In [ ]:
!mkdir -p /content/drive/MyDrive/yolo_backup

!cp -r /content/foggy-car-1 /content/drive/MyDrive/yolo_backup/
!cp -r /content/runs /content/drive/MyDrive/yolo_backup/


In [ ]:
!ls /content/drive/MyDrive/yolo_backup


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

img = Image.open("/content/runs/detect/train/results.png")
plt.imshow(img)
plt.show()

In [ ]:
cm =  Image.open("/content/runs/detect/train/confusion_matrix.png")
plt.imshow(cm)
plt.show()

In [ ]:
Image.open("/content/runs/detect/train/val_batch0_labels.jpg")

In [ ]:
!ffmpeg -i /content/46_hazy_video.mp4 \
-vcodec libx264 -crf 28 -preset slow \
-acodec aac -b:a 128k \
/content/46_hazy_video_compressed.mp4


In [ ]:
from ultralytics import YOLO

model = YOLO("/content/runs/detect/train/weights/best.pt")

results = model.predict(
    source="/content/46_hazy_video_compressed.mp4",
    imgsz=640,
    conf=0.2,
    save=True,
    device=0
)


In [ ]:
!ls /content/runs/detect


In [ ]:
!ffmpeg -i /content/runs/detect/predict/46_hazy_video_compressed.avi \
-vcodec libx264 -crf 23 -preset fast \
/content/output_fixed.mp4


In [ ]:
from IPython.display import Video
Video("/content/output_fixed.mp4", embed=True)


In [ ]:
!mkdir -p /content/drive/MyDrive/yolo26_runs
!cp -r /content/runs /content/drive/MyDrive/yolo26_runs/


In [ ]:
!rsync -av /content/runs /content/drive/MyDrive/yolo_backup/


In [ ]:
!ls /content/drive/MyDrive/yolo_backup/foggy-car-1


In [ ]:
!ls /content/drive/MyDrive/yolo_backup/foggy-car-1/test


In [ ]:
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/yolo_backup/runs/detect/train/weights/best.pt")

results = model.predict(
    source="/content/drive/MyDrive/yolo_backup/foggy-car-1/test/images",
    imgsz=640,
    conf=0.2,
    save=True
)


In [ ]:
metrics = model.val(
    data="/content/drive/MyDrive/yolo_backup/foggy-car-1/data.yaml",
    split="test",
    imgsz=640
)


In [ ]:
!zip -r predict_results.zip /content/runs/detect/predict


In [ ]:
from google.colab import files
files.download("predict_results.zip")


In [ ]:
!zip -r val_results.zip /content/runs/detect/val


In [ ]:
from google.colab import files
files.download("val_results.zip")
